In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "health_and_habit_guardian"
GOLD_DAILY = f"{CATALOG}.gold.daily_health"
GOLD_WEEKLY = f"{CATALOG}.gold.weekly_trends"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [0]:
weekly = (
    spark.table(GOLD_DAILY)
    .withColumn("week_start", F.to_date(F.date_trunc("week", F.col("date"))))
    .groupBy("user_id", "week_start")
    .agg(
        F.avg("steps").alias("avg_steps"),
        F.avg("active_minutes").alias("avg_active_minutes"),
        F.avg("sleep_hours").alias("avg_sleep_hours"),
        F.avg("sleep_efficiency").alias("avg_sleep_efficiency"),
        F.avg("resting_hr").alias("avg_resting_hr"),
        F.avg("recovery_score").alias("avg_recovery_score"),
        F.sum("sleep_debt_hours").alias("sleep_debt"),
        F.count("*").alias("days_observed"),
    )
)

w = Window.partitionBy("user_id").orderBy("week_start")

weekly = (
    weekly
    .withColumn("prev_avg_steps", F.lag("avg_steps").over(w))
    .withColumn("prev_avg_recovery", F.lag("avg_recovery_score").over(w))
    .withColumn(
        "activity_change_pct",
        F.when(
            F.col("prev_avg_steps").isNull() | (F.col("prev_avg_steps") == 0),
            F.lit(None).cast("double"),
        ).otherwise((F.col("avg_steps") - F.col("prev_avg_steps")) / F.col("prev_avg_steps")),
    )
    .withColumn(
        "recovery_change_pct",
        F.when(
            F.col("prev_avg_recovery").isNull() | (F.col("prev_avg_recovery") == 0),
            F.lit(None).cast("double"),
        ).otherwise(
            (F.col("avg_recovery_score") - F.col("prev_avg_recovery")) / F.col("prev_avg_recovery")
        ),
    )
    .drop("prev_avg_steps", "prev_avg_recovery")
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
final_cols = [
    "user_id", "week_start",
    "avg_steps", "avg_active_minutes", "avg_sleep_hours",
    "avg_sleep_efficiency", "avg_resting_hr", "avg_recovery_score",
    "sleep_debt", "days_observed",
    "activity_change_pct", "recovery_change_pct",
    "_processed_at",
]

weekly_final = weekly.select(*final_cols)
weekly_final.createOrReplaceTempView("weekly_updates")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_WEEKLY}
USING DELTA
AS SELECT * FROM weekly_updates WHERE 1 = 0
""")

cols = ", ".join(final_cols)
spark.sql(f"""
MERGE INTO {GOLD_WEEKLY} AS target
USING (
    SELECT {cols}
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY user_id, week_start
            ORDER BY _processed_at DESC NULLS LAST
        ) AS _rn
        FROM weekly_updates
    )
    WHERE _rn = 1
) AS source
ON target.user_id = source.user_id AND target.week_start = source.week_start
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"SELECT COUNT(*) AS rows, MIN(week_start), MAX(week_start) FROM {GOLD_WEEKLY}"))

rows,MIN(week_start),MAX(week_start)
28,2026-07-27,2026-09-07
